# 🇵🇰 Pakistan Climate Extremes — Hybrid Pipeline
## Step 1: Extract Pakistan (Python) → Step 2: Calculate Extremes (CDO) → Step 3: Plot (Python)

---

###  Pakistan Bounding Box
We use the following coordinates to crop our global datasets:

| Parameter | Value |
|-----------|-------|
| West Longitude | 60° E |
| East Longitude | 78° E |
| South Latitude | 23° N |
| North Latitude | 37° N |

**Python Syntax:** `ds.sel(lon=slice(60, 78), lat=slice(37, 23))`

---

In [ ]:
import subprocess
import os
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# ============================================================
#  SET YOUR FULL INPUT FILE PATHS HERE
# ============================================================
tasmax_global = "/mnt/sdb2/WCS-S2_Data/ObservationalData/CPC/tasmax/tasmax_global_daily_gridded_obs_cpc_1979-2025.nc"
tasmin_global = "/mnt/sdb2/WCS-S2_Data/ObservationalData/CPC/tasmin/tasmin_global_daily_gridded_obs_cpc_1979-2024.nc"
pr_global     = "/mnt/sdb2/WCS-S2_Data/ObservationalData/CPC/pr/pr_global_daily_gridded_obs_cpc_1979-2024.nc"  # Fix path!

# Pakistan bounding box
LON_W, LON_E, LAT_S, LAT_N = 60, 78, 23, 37

print(" Libraries loaded. Ready to begin!")

---
## 🗺️ STEP 1: Extract Pakistan using Python (`xarray`)

Instead of using CDO, we will load the data in Python and use `.sel(lat=slice(), lon=slice())` to crop it. 
Then, we will save these smaller Pakistan files to disk using `.to_netcdf()` so CDO can use them in the next step.

**🚨 CRITICAL CLIMATE PYTHON RULE:** 
Since our CPC dataset stores latitudes from North to South (e.g., 89 to -89), we **MUST** slice the latitude backwards: `slice(LAT_N, LAT_S)`. If we do it the normal way `slice(LAT_S, LAT_N)`, xarray will give us an empty map!

In [ ]:
# Output files after cropping
tasmax_pak = "pak_tasmax.nc"
tasmin_pak = "pak_tasmin.nc"
pr_pak     = "pak_pr.nc"

def extract_pakistan_with_python(global_file, pak_file, name):
    if not os.path.exists(pak_file):
        print(f" Loading {name} global file and extracting Pakistan using Python...")
        try:
            # 1. Load the huge global dataset (dask helps if the file is massive)
            ds = xr.open_dataset(global_file, chunks={'time': 100})
            
            # 2. Extract Pakistan using .sel()
            # Notice how latitude is backwards! 37 to 23.
            ds_pak = ds.sel(lon=slice(LON_W, LON_E), lat=slice(LAT_N, LAT_S))
            
            # 3. Save the tiny cropped dataset back to disk for CDO to read
            ds_pak.to_netcdf(pak_file)
            ds.close()
            ds_pak.close()
            print(f"    Done! Saved as: {pak_file}")
            
        except FileNotFoundError:
            print(f"    Error: Could not find {global_file}. Check your input path!")
        except Exception as e:
            print(f"    Error processing {name}: {e}")
    else:
        print(f"    {pak_file} already exists. Skipping Python extraction.")

# Run the extraction for all 3 datasets
extract_pakistan_with_python(tasmax_global, tasmax_pak, "tasmax")
extract_pakistan_with_python(tasmin_global, tasmin_pak, "tasmin")
extract_pakistan_with_python(pr_global, pr_pak, "pr")

print("\n All datasets extracted to Pakistan using Python!")

---
## 🔬 STEP 2: Verify the Cropped Pakistan Data
Let's quickly load one of the cropped files to confirm the coordinates are correct.

In [ ]:
try:
    ds_check = xr.open_dataset(tasmax_pak)
    print(" Pakistan tasmax dataset info:")
    print(ds_check)
    print(f"\n Lat range: {float(ds_check.lat.min()):.2f} to {float(ds_check.lat.max()):.2f}")
    print(f" Lon range: {float(ds_check.lon.min()):.2f} to {float(ds_check.lon.max()):.2f}")
    ds_check.close()
except FileNotFoundError:
    print("File not created yet.")

---
## 🌡️ STEP 3: Calculate Temperature Extremes with CDO

Now that Python has prepared the tiny `pak_*.nc` files, we use Python's `subprocess` to trigger **CDO** to calculate the extremes incredibly fast.

### 3A. TXx — Hottest Day of the Year (`yearmax` on tasmax)

In [ ]:
txx_file = "pak_TXx.nc"

if not os.path.exists(txx_file):
    print(" Calculating TXx (Annual Maximum of tasmax)...")
    subprocess.run(["cdo", "yearmax", tasmax_pak, txx_file], check=True)
    print("TXx calculated!")
else:
    print(f"{txx_file} already exists.")

### 3B. TXn — Coldest Day of the Year (`yearmin` on tasmax)

In [ ]:
txn_file = "pak_TXn.nc"

if not os.path.exists(txn_file):
    print("Calculating TXn (Annual Minimum of tasmax)...")
    subprocess.run(["cdo", "yearmin", tasmax_pak, txn_file], check=True)
    print("TXn calculated!")
else:
    print(f" {txn_file} already exists.")

### 3C. TNx — Warmest Night of the Year (`yearmax` on tasmin)

In [ ]:
tnx_file = "pak_TNx.nc"

if not os.path.exists(tnx_file):
    print("Calculating TNx (Annual Maximum of tasmin)...")
    subprocess.run(["cdo", "yearmax", tasmin_pak, tnx_file], check=True)
    print("TNx calculated!")
else:
    print(f"{tnx_file} already exists.")

### 3D. TNn — Coldest Night of the Year (`yearmin` on tasmin)

In [ ]:
tnn_file = "pak_TNn.nc"

if not os.path.exists(tnn_file):
    print("Calculating TNn (Annual Minimum of tasmin)...")
    subprocess.run(["cdo", "yearmin", tasmin_pak, tnn_file], check=True)
    print("TNn calculated!")
else:
    print(f"{tnn_file} already exists.")

### 3E. TX90p — Warm Days (`eca_tx90p` on tasmax)

In [ ]:
tx90p_file = "pak_TX90p.nc"

if not os.path.exists(tx90p_file):
    print("Calculating TX90p (Warm Days)...")
    subprocess.run(["cdo", "eca_tx90p", tasmax_pak, tx90p_file], check=True)
    print("TX90p calculated!")
else:
    print(f"{tx90p_file} already exists.")

### 3F. TN10p — Cold Nights (`eca_tn10p` on tasmin)

In [ ]:
tn10p_file = "pak_TN10p.nc"

if not os.path.exists(tn10p_file):
    print("Calculating TN10p (Cold Nights)...")
    subprocess.run(["cdo", "eca_tn10p", tasmin_pak, tn10p_file], check=True)
    print("TN10p calculated!")
else:
    print(f"{tn10p_file} already exists.")

---
## 🌧️ STEP 4: Calculate Precipitation Extremes with CDO

### 4A. Rx1day — Maximum 1-Day Rainfall (`eca_rx1day` on pr)

In [ ]:
rx1day_file = "pak_Rx1day.nc"

if not os.path.exists(rx1day_file):
    print("Calculating Rx1day (Max 1-Day Rainfall)...")
    subprocess.run(["cdo", "eca_rx1day", pr_pak, rx1day_file], check=True)
    print("Rx1day calculated!")
else:
    print(f"{rx1day_file} already exists.")

### 4B. Rx5day — Maximum 5-Day Rainfall (`eca_rx5day` on pr)

In [ ]:
rx5day_file = "pak_Rx5day.nc"

if not os.path.exists(rx5day_file):
    print("Calculating Rx5day (Max 5-Day Rainfall)...")
    subprocess.run(["cdo", "eca_rx5day", pr_pak, rx5day_file], check=True)
    print("Rx5day calculated!")
else:
    print(f"{rx5day_file} already exists.")

---
## 🗺️ STEP 5: Plot All Extremes — Pakistan Spatial Maps

A helper function to plot any extreme index as a map over Pakistan.

In [ ]:
def plot_extreme(nc_file, title, cmap, unit_label):
    """Load a CDO output file and plot the long-term mean as a Pakistan map."""
    if not os.path.exists(nc_file):
        print(f"File {nc_file} missing!")
        return
        
    ds = xr.open_dataset(nc_file)
    
    # Automatically find the correct 2D variable (ignores time_bnds)
    var_name = [v for v in ds.data_vars if 'lat' in ds[v].dims][0]
    data = ds[var_name].mean(dim="time")
    
    fig = plt.figure(figsize=(9, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    ax.set_extent([LON_W, LON_E, LAT_S, LAT_N], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.BORDERS, linestyle="-", linewidth=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.STATES, linestyle=":", linewidth=0.5)
    
    img = data.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap=cmap, add_colorbar=True,
        cbar_kwargs={"label": unit_label, "shrink": 0.8}
    )
    ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
    plt.title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    ds.close()

print("Plot function ready!")

In [ ]:
# --- Plot all extremes one by one ---
plot_extreme(txx_file,    "TXx — Mean Annual Hottest Day (Pakistan)",          "YlOrRd",  "Temperature (°C)")
plot_extreme(txn_file,    "TXn — Mean Annual Coldest Day (Pakistan)",          "RdYlBu",  "Temperature (°C)")
plot_extreme(tnx_file,    "TNx — Mean Annual Warmest Night (Pakistan)",        "OrRd",    "Temperature (°C)")
plot_extreme(tnn_file,    "TNn — Mean Annual Coldest Night (Pakistan)",        "Blues",   "Temperature (°C)")
plot_extreme(tx90p_file,  "TX90p — Warm Days Frequency (Pakistan)",            "Reds",    "% of Days")
plot_extreme(tn10p_file,  "TN10p — Cold Nights Frequency (Pakistan)",          "PuBu",    "% of Days")
plot_extreme(rx1day_file, "Rx1day — Mean Annual Max 1-Day Rainfall (Pakistan)","GnBu",    "Precipitation (mm)")
plot_extreme(rx5day_file, "Rx5day — Mean Annual Max 5-Day Rainfall (Pakistan)","Blues",   "Precipitation (mm)")